In [5]:
import os

from IPython.display import FileLink

if os.getcwd() == '/notebooks':
    os.chdir("./motion-synthesis")
    print('inside dir: ', os.listdir())

print("Click here to download the motion-dataset.zip: ", FileLink("motion-dataset.zip"))


Click here to download the motion-dataset.zip:  /notebooks/motion-synthesis/motion-dataset.zip


In [6]:
import torch
from options.train_options  import TrainOptions
from os.path import join as pjoin
import os
from utils.paramUtils import t2m_kinematic_chain
import numpy as np
from utils.word_vectorizer import WordVectorizer
from torch.utils.data import DataLoader
from data_utils.dataset import MotionDatasetV2
from data_utils.dataset import PartMotionDatasetV2
from networks.nn import MotionVQVAE
from networks.trainers import MotionVQVAETrainer
from torch.utils.data import Subset


In [ ]:
parser = TrainOptions()
options = parser.parse(args = ['--max_epoch', '1000'])
options.gpu_id = torch.cuda.current_device() if torch.cuda.is_available() else -1
options.device = torch.device("cpu" if options.gpu_id==-1 else "cuda:" + str(options.gpu_id))
torch.autograd.set_detect_anomaly(True)

if options.gpu_id != -1:
    # self.opt.gpu_id = int(self.opt.gpu_id)
    torch.cuda.set_device(options.gpu_id)

print('\nDevice used: ', options.device)

options.save_root = pjoin(options.checkpoints_dir, 'HumanML3D', options.name)
options.model_dir = pjoin(options.checkpoints_dir, 'model')
options.meta_dir = pjoin(options.save_root, 'meta')
options.eval_dir = pjoin(options.save_root, 'animation')
options.log_dir = pjoin('./log', options.dataset_name, options.name)
options.save_every_e = 100
options.is_continue = False

os.makedirs(options.model_dir, exist_ok=True)
os.makedirs(options.meta_dir, exist_ok=True)
os.makedirs(options.eval_dir, exist_ok=True)
os.makedirs(options.log_dir, exist_ok=True)

options.data_root = './data/HumanML3D'
options.motion_dir = pjoin(options.data_root, 'new_joint_vecs')
options.text_dir = pjoin(options.data_root, 'texts')
options.joints_num = 22
options.max_motion_length = 196
dim_pose = 263
radius = 4
fps = 20
kinematic_chain = t2m_kinematic_chain


Device used:  cuda:0


In [8]:
mean = np.load(pjoin(options.data_root, 'Mean.npy'))
std = np.load(pjoin(options.data_root, 'Std.npy'))

w_vectorizer = WordVectorizer('./glove', 'our_vab')
train_split_file = pjoin(options.data_root, 'train_micro.txt')
val_split_file = pjoin(options.data_root, 'val_micro.txt')

if options.dataset_mode == "micro":
    par_train_dataset = PartMotionDatasetV2(options, mean, std, train_split_file)
    par_val_dataset = PartMotionDatasetV2(options, mean, std, val_split_file)
    all_train_indices = np.arange(len(par_train_dataset))
    micro_train_indices = all_train_indices[:80]
    all_val_indices = np.arange(len(par_val_dataset))
    micro_val_indices = all_val_indices[:30]
    train_dataset = Subset(par_train_dataset, micro_train_indices)
    val_dataset = Subset(par_val_dataset, micro_val_indices)
else:
    train_dataset = PartMotionDatasetV2(options, mean, std, train_split_file)
    val_dataset = PartMotionDatasetV2(options, mean, std, val_split_file)

print('\nTrain Part dataset length: ', len(train_dataset))
sample_motion = train_dataset[-1]
print('Sample data shape: ', sample_motion['motion_parts'].shape)
Dp_max = sample_motion['motion_parts'].shape[-1]

id list 8


100%|██████████| 8/8 [00:00<00:00, 7356.81it/s]


Motion shape (B, T, D): (8, 199, 263)
Total number of motions 8, snippets 468
id list 4


100%|██████████| 4/4 [00:00<00:00, 4673.32it/s]

Motion shape (B, T, D): (4, 170, 263)
Total number of motions 4, snippets 495

Train Part dataset length:  80
Sample data shape:  (40, 6, 60)


In [9]:
train_loader = DataLoader(train_dataset, batch_size=options.batch_size, drop_last=not(options.dataset_mode == "micro"), num_workers=1,
                              shuffle=False, pin_memory=True)
val_loader = DataLoader(train_dataset, batch_size=options.batch_size, drop_last=not(options.dataset_mode == "micro"), num_workers=1,
                        shuffle=False, pin_memory=True)
vqvae = MotionVQVAE(
    input_dim=Dp_max,
    enc_hidden_dim=1024,
    dec_hidden_dim=1024,
    latent_dim=256,
    num_embeddings=512,
    beta=0.1
)

trainer = MotionVQVAETrainer(options, vqvae = vqvae)
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader)


Number of epochs: 10000
Iters Per Epoch, Training: 0001, Validation: 001
Epoch: 0
Train Loss: 1.89459 Reconstruction Loss: 0.94717 VQ Loss: 0.94741 Codebook Loss: 0.86128 Commitment Loss: 0.86128
Validation Loss: 0.60342 Reconstruction Loss: 0.45618 VQ Loss: 0.14724 Codebook Loss: 0.13385 Commitment Loss: 0.13385
Epoch: 100
Train Loss: 0.87390 Reconstruction Loss: 0.69304 VQ Loss: 0.18086 Codebook Loss: 0.16442 Commitment Loss: 0.16442
Validation Loss: 0.40242 Reconstruction Loss: 0.34434 VQ Loss: 0.05807 Codebook Loss: 0.05279 Commitment Loss: 0.05279
Epoch: 200
Train Loss: 0.81971 Reconstruction Loss: 0.43409 VQ Loss: 0.38562 Codebook Loss: 0.35056 Commitment Loss: 0.35056
Validation Loss: 0.35239 Reconstruction Loss: 0.21143 VQ Loss: 0.14096 Codebook Loss: 0.12814 Commitment Loss: 0.12814
Epoch: 300
Train Loss: 0.58046 Reconstruction Loss: 0.33067 VQ Loss: 0.24979 Codebook Loss: 0.22708 Commitment Loss: 0.22708
Validation Loss: 0.24739 Reconstruction Loss: 0.15708 VQ Loss: 0.09031 C